In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
from collections import defaultdict

def plot_participant_metrics(participant_dir):
    """
    Reads TensorFlow event files and hparams.yaml from a directory,
    prints hyperparameters, and plots all scalar metrics in separate plots.

    Args:
        participant_dir (str): The path to the participant's directory
                               (e.g., 'metrics/participant_0').
    """
    print(f"--- Processing directory: {participant_dir} ---")

    # --- 1. Read and print hyperparameters ---
    hparams_path = os.path.join(participant_dir, 'hparams.yaml')
    try:
        with open(hparams_path, 'r') as f:
            hparams = yaml.safe_load(f)
            print("Hyperparameters found:")
            print(yaml.dump(hparams, indent=2))
    except FileNotFoundError:
        print("hparams.yaml not found in this directory.")
    except Exception as e:
        print(f"Error reading hparams.yaml: {e}")

    # --- 2. Load data from .tfevents files ---
    try:
        ea = event_accumulator.EventAccumulator(
            participant_dir,
            size_guidance={event_accumulator.SCALARS: 0} # Load all scalars
        )
        ea.Reload() # Load all events from disk
    except Exception as e:
        print(f"Error loading event files: {e}")
        print("Please ensure TensorFlow is installed ('pip install tensorflow').")
        return

    scalar_tags = ea.Tags()['scalars']
    if not scalar_tags:
        print("No scalar metrics found in the event files.")
        return

    print(f"\nFound scalar tags: {scalar_tags}")

    # --- 3. Plot each scalar metric ---
    for tag in scalar_tags:
        events = ea.Scalars(tag)
        steps = [event.step for event in events]
        values = [event.value for event in events]

        plt.figure(figsize=(10, 6))
        plt.plot(steps, values, label=tag)
        plt.title(f'Metric: {tag} for {os.path.basename(participant_dir)}')
        plt.xlabel('Step')
        plt.ylabel('Value')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

    print("\nDisplaying plots...")
    plt.show()
    print("--- Finished processing. ---")


def plot_multiple_participants(participant_dirs):
    """
    Reads TensorFlow event files from multiple participant directories
    and plots a comparison for each scalar metric on a shared plot.

    Args:
        participant_dirs (list[str]): A list of paths to participant directories.
    """
    all_metrics = {}  # Structure: {tag: {participant_name: (steps, values)}}

    print("--- Processing multiple participant directories ---")

    # --- 1. Collect data from all participants ---
    for participant_dir in participant_dirs:
        participant_name = os.path.basename(participant_dir)
        print(f"Reading data from: {participant_name}")

        try:
            ea = event_accumulator.EventAccumulator(
                participant_dir,
                size_guidance={event_accumulator.SCALARS: 0}
            )
            ea.Reload()

            scalar_tags = ea.Tags()['scalars']
            if not scalar_tags:
                print(f"  - No scalar metrics found for {participant_name}.")
                continue

            for tag in scalar_tags:
                events = ea.Scalars(tag)
                steps = [e.step for e in events]
                values = [e.value for e in events]

                if tag not in all_metrics:
                    all_metrics[tag] = {}
                all_metrics[tag][participant_name] = (steps, values)

        except Exception as e:
            print(f"  - Error loading event files for {participant_name}: {e}")
            continue

    if not all_metrics:
        print("\nNo metrics were found across any of the provided directories.")
        return

    # --- 2. Plot collected data, one plot per metric tag ---
    print("\n--- Generating comparison plots ---")
    for tag, participant_data in all_metrics.items():
        plt.figure(figsize=(12, 7))
        plt.title(f'Metric Comparison: {tag}')
        plt.xlabel('Step')
        plt.ylabel('Value')

        for participant_name, (steps, values) in participant_data.items():
            plt.plot(steps, values, label=participant_name, alpha=0.8)

        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.legend()
        plt.tight_layout()

    print("\nDisplaying plots...")
    plt.show()
    print("--- Finished processing. ---")


def plot_experiment_comparison(experiment_dirs):
    """
    Compares multiple experiments by plotting an aggregated metric from their participants.

    For each experiment and each metric tag, it calculates a single line representing
    the best performance (max or min) across all its participants at each step.
    Then, it plots these lines from all experiments on a single graph for comparison.

    Args:
        experiment_dirs (list[str]): A list of paths to experiment directories.
    """
    all_experiments_metrics = {}  # {tag: {exp_name: (steps, agg_values)}}

    print("--- Processing and comparing multiple experiments ---")

    # --- 1. Loop through each experiment directory ---
    for exp_dir in experiment_dirs:
        exp_name = os.path.basename(exp_dir)
        metrics_dir = os.path.join(exp_dir, 'metrics')

        if not os.path.isdir(metrics_dir):
            print(f"  - Skipping '{exp_name}': 'metrics' directory not found.")
            continue

        print(f"Processing experiment: {exp_name}")

        participant_dirs = [
            os.path.join(metrics_dir, d)
            for d in os.listdir(metrics_dir)
            if os.path.isdir(os.path.join(metrics_dir, d)) and d.startswith('participant_')
        ]

        if not participant_dirs:
            print(f"  - No participant directories found in '{metrics_dir}'.")
            continue

        # --- 2. Collect data from all participants within this experiment ---
        experiment_data_by_tag = defaultdict(list)
        for p_dir in participant_dirs:
            try:
                ea = event_accumulator.EventAccumulator(p_dir, size_guidance={event_accumulator.SCALARS: 0})
                ea.Reload()
                for tag in ea.Tags()['scalars']:
                    events = ea.Scalars(tag)
                    steps = [e.step for e in events]
                    values = [e.value for e in events]
                    experiment_data_by_tag[tag].append((steps, values))
            except Exception as e:
                p_name = os.path.basename(p_dir)
                print(f"    - Error loading data for {p_name}: {e}")

        # --- 3. Aggregate participant data for this experiment ---
        for tag, participant_runs in experiment_data_by_tag.items():
            step_to_values = defaultdict(list)
            for steps, values in participant_runs:
                for step, value in zip(steps, values):
                    step_to_values[step].append(value)

            if not step_to_values: continue

            sorted_steps = sorted(step_to_values.keys())
            agg_func = min if 'loss' in tag.lower() else max
            aggregated_values = [agg_func(step_to_values[step]) for step in sorted_steps]

            if tag not in all_experiments_metrics:
                all_experiments_metrics[tag] = {}
            all_experiments_metrics[tag][exp_name] = (sorted_steps, aggregated_values)

    if not all_experiments_metrics:
        print("\nNo metrics were found across any of the experiments.")
        return

    # --- 4. Plot the aggregated data for all experiments ---
    print("\n--- Generating experiment comparison plots ---")
    for tag, experiments_data in all_experiments_metrics.items():
        plt.figure(figsize=(12, 7))
        agg_type = "MIN" if 'loss' in tag.lower() else "MAX"
        plt.title(f'Experiment Comparison: Best Participant Performance ({agg_type}) for "{tag}"')
        plt.xlabel('Step')
        plt.ylabel('Aggregated Value')

        for exp_name, (steps, values) in experiments_data.items():
            plt.plot(steps, values, label=exp_name, alpha=0.9)

        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.legend()
        plt.tight_layout()

    print("\nDisplaying plots...")
    plt.show()
    print("--- Finished processing. ---")



In [ ]:
import glob
import os

base_path = '../app/logs/'
search_pattern = os.path.join(base_path, 'nebula_DFL_*')

print(f"Searching for directories with pattern: {search_pattern}")

# Use glob.glob() to find all files and directories matching the pattern.
all_matching_paths = glob.glob(search_pattern)

# Filter the list to include only directories, not files.
# os.path.isdir() checks if a given path is a directory.
experiment_paths = [path for path in all_matching_paths if os.path.isdir(path)]

# Sort the paths for consistency (optional, but good practice).
experiment_paths.sort()

# Print the list of found experiment directories.
print("\nFound experiment directories:")
print(f"Total: {len(experiment_paths)}")
# if experiment_paths:
#     for path in experiment_paths:
#         print(path)
# else:
#     print("No matching directories found.")


In [17]:

if __name__ == '__main__':
    # experiment_paths = [
    #     '../app/logs/nebula_DFL_2025_10_15_20_17_14', # <--- IMPORTANT: CHANGE THIS
    #     '../app/logs/nebula_DFL_2025_10_15_20_33_05'  # <--- IMPORTANT: CHANGE THIS
    # ]

    valid_experiment_paths = [p for p in experiment_paths if os.path.isdir(p)]

    if not experiment_paths or not valid_experiment_paths:
        print("="*50)
        print("WARNING: No valid experiment directories were provided.")
        print("Please update the 'experiment_paths' list in the script.")
        print(f"Example paths given: {experiment_paths}")
        print("="*50)
    else:
        if len(valid_experiment_paths) < len(experiment_paths):
            print("="*50)
            print("WARNING: Some provided paths were not found and will be skipped.")
            print(f"Found: {valid_experiment_paths}")
            print(f"Not found: {[p for p in experiment_paths if p not in valid_experiment_paths]}")
            print("="*50)
        plot_experiment_comparison(valid_experiment_paths)


    # --- USAGE EXAMPLE for multiple participants in ONE experiment ---
    # To plot participants from a single experiment, uncomment the block below.
    #
    # paths_to_data = [
    #     'path/to/your/experiment/metrics/participant_0',
    #     'path/to/your/experiment/metrics/participant_1'
    # ]
    # valid_paths = [p for p in paths_to_data if os.path.isdir(p)]
    # if valid_paths:
    #     plot_multiple_participants(valid_paths)
    # else:
    #     print("WARNING: No valid participant paths provided for single experiment plotting.")

    # --- USAGE EXAMPLE for a single participant ---
    # To plot just one participant, uncomment the block below.
    #
    # single_path = 'path/to/your/participant_0' # <--- CHANGE THIS
    # if os.path.isdir(single_path):
    #     plot_participant_metrics(single_path)
    # else:
    #     print(f"WARNING: The example path '{single_path}' does not exist.")

